# Bài 2 — Phân khúc khách hàng

Thực hiện theo mục 3. Phân cụm theo thu nhập và điểm chi tiêu; dùng tuổi để mô tả từng nhóm.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/khach_hang.csv")
print("Kích thước:", df.shape)
display(df.describe().round(1))

## Chuẩn hóa và phân cụm
Hai đặc trưng có thang đo khác nhau nên chuẩn hóa trước. Tâm cụm được chuyển về đơn vị gốc để dễ diễn giải.

In [ ]:
X = df[["thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
df["cum"] = km.labels_
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=X.columns)
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby("cum")["tuoi"].mean().reindex(range(5)).values
display(tam.round(1))
print(f"Inertia: {km.inertia_:.2f}")

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df.thu_nhap, df.diem_chi_tieu, c=df.cum, cmap="tab10", alpha=.75)
centers = sc.inverse_transform(km.cluster_centers_)
plt.scatter(centers[:,0], centers[:,1], c="black", marker="X", s=180, label="Tâm cụm")
plt.xlabel("Thu nhập (triệu đồng/tháng)"); plt.ylabel("Điểm chi tiêu"); plt.title("Phân khúc khách hàng (K=5)"); plt.legend(); plt.show()

## Xếp khách hàng mới
Dùng chính bộ chuẩn hóa đã fit ở trên trước khi gọi `predict`.

In [ ]:
moi = pd.DataFrame({"thu_nhap": [29.0, 9.0], "diem_chi_tieu": [85, 75]})
cum_moi = km.predict(sc.transform(moi))
moi.assign(cum=cum_moi)